# 🟢 Solution: Batch AABB Overlap

**Primitive:** broadcasting comparison (Separating Axis Theorem)

**Reduction:** `out[i,j]` is True iff `a[i]` and `b[j]` are not separated on either axis — checked via four strict inequality comparisons broadcast over the N×M grid.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import numpy as np

In [ ]:
# primitive: broadcasting comparison

import numpy as np

def rect_overlap(rects_a, rects_b):
    a = rects_a[:, None, :]  # (N, 1, 4)
    b = rects_b[None, :, :]  # (1, M, 4)
    x_overlap = (a[..., 0] < b[..., 2]) & (a[..., 2] > b[..., 0])
    y_overlap = (a[..., 1] < b[..., 3]) & (a[..., 3] > b[..., 1])
    return x_overlap & y_overlap

In [ ]:
# 🔍 Verify solution
rects_a = np.array([[0.0, 0.0, 2.0, 2.0],
                    [5.0, 5.0, 7.0, 7.0]])
rects_b = np.array([[1.0, 1.0, 3.0, 3.0],
                    [6.0, 0.0, 8.0, 2.0]])
result = rect_overlap(rects_a, rects_b)
print("shape:", result.shape)        # expect (2, 2)
print("result:\n", result)
# expect: [[True, False], [False, False]]

In [ ]:
# ✅ Inline test suite
import numpy as np, time

# ── Test 1: basic ──────────────────────────────────────────────────────────
rects_a = np.array([[0.0,0.0,2.0,2.0],[5.0,5.0,7.0,7.0]])
rects_b = np.array([[1.0,1.0,3.0,3.0],[6.0,0.0,8.0,2.0]])
r = rect_overlap(rects_a, rects_b)
assert r.shape == (2,2), f"Shape: {r.shape}"
assert r[0,0]==True and r[0,1]==False and r[1,0]==False and r[1,1]==False, f"Result: {r}"
print("Test 1 passed: basic overlapping / non-overlapping")

# ── Test 2: separated on X ─────────────────────────────────────────────────
a = np.array([[0.0,0.0,1.0,1.0]]); b = np.array([[2.0,0.0,3.0,1.0]])
assert rect_overlap(a,b)[0,0]==False, "Separated on X should be False"
print("Test 2 passed: separation on X")

# ── Test 3: separated on Y ─────────────────────────────────────────────────
a = np.array([[0.0,0.0,1.0,1.0]]); b = np.array([[0.0,2.0,1.0,3.0]])
assert rect_overlap(a,b)[0,0]==False, "Separated on Y should be False"
print("Test 3 passed: separation on Y")

# ── Test 4: touching at edge (strict > → False) ────────────────────────────
a = np.array([[0.0,0.0,1.0,1.0]]); b = np.array([[1.0,0.0,2.0,1.0]])
assert rect_overlap(a,b)[0,0]==False, "Touching at edge — strict > means False"
print("Test 4 passed: touching edge is not overlap")

# ── Test 5: large N=M=3000 ─────────────────────────────────────────────────
rng = np.random.default_rng(0)
xy = rng.uniform(0,100,(3000,2)); wh = rng.uniform(1,10,(3000,2))
ra = np.concatenate([xy, xy+wh], axis=1)
xy = rng.uniform(0,100,(3000,2)); wh = rng.uniform(1,10,(3000,2))
rb = np.concatenate([xy, xy+wh], axis=1)
t0 = time.time(); result = rect_overlap(ra,rb); elapsed = time.time()-t0
assert result.shape==(3000,3000), f"Shape: {result.shape}"
assert elapsed < 5.0, f"Too slow: {elapsed:.2f}s"
print(f"Test 5 passed: large N=M=3000 ({elapsed:.3f}s)")

print("\nAll tests passed!")